# PDFD (padronizado)


## 1) Importação do dataset e bibliotecas


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15


## 2) Execução do `setup()` e alinhamento temporal


In [ ]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    save_artifacts=False,
)

processed_df = raw_df.reset_index().copy()
if "Change %" in processed_df.columns:
    processed_df["Change %"] = (
        processed_df["Change %"].str.replace("%", "", regex=False).astype(float) / 100
    )

processed_df = processed_df.sort_values("Date")
processed_df["target_return"] = (
    processed_df[TARGET_COL].pct_change(HORIZON).shift(-HORIZON)
)
processed_df = processed_df.dropna().set_index("Date")

n_rows = len(processed_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
val_index = processed_df.iloc[train_end:val_end].index

print("Validation length:", len(val_index))


## 3) Construção da ferramenta de risco (PDFD + momentos)


In [ ]:
returns = processed_df[TARGET_COL].pct_change().fillna(0.0)
risk_window = 252

pdfd_dataset = pd.DataFrame(index=val_index)
pdfd_dataset["pdfd_005"] = (returns <= -0.005).rolling(risk_window, min_periods=30).mean().reindex(val_index)
pdfd_dataset["pdfd_01"] = (returns <= -0.01).rolling(risk_window, min_periods=30).mean().reindex(val_index)
pdfd_dataset["pdfd_02"] = (returns <= -0.02).rolling(risk_window, min_periods=30).mean().reindex(val_index)
pdfd_dataset["pdfd_03"] = (returns <= -0.03).rolling(risk_window, min_periods=30).mean().reindex(val_index)

pdfd_dataset["pdf_skew"] = returns.rolling(risk_window, min_periods=30).skew().reindex(val_index)
pdfd_dataset["pdf_kurtosis"] = returns.rolling(risk_window, min_periods=30).kurt().reindex(val_index)

pdfd_dataset = pdfd_dataset.bfill().sort_index()


## 4) Sanity checks mínimos


In [ ]:
print("NaN críticos:")
print(pdfd_dataset.isna().sum())

print("\nDistribuição plausível:")
print(pdfd_dataset.describe().T[["mean", "std", "min", "max"]])

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", pdfd_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", pdfd_dataset.index.has_duplicates)

print("Dataset lines == val_index lines:", len(pdfd_dataset) == len(val_index))
print("Dataset index == val_index:", pdfd_dataset.index.equals(val_index))


## 5) DataFrame final (features de risco) e 6) Padronização


In [ ]:
pdfd_dataset = pdfd_dataset[["pdfd_005", "pdfd_01", "pdfd_02", "pdfd_03", "pdf_skew", "pdf_kurtosis"]]

pdfd_dataset.head(), pdfd_dataset.shape


## 7) Salvamento (opcional)


In [ ]:
# pdfd_dataset.to_parquet("pdfd_dataset.parquet", index=True)
